# Manual topic labelling — Archelec / BERTopic ($K=30$)

Ce notebook documente le travail manuel de labellisation des topics produits par BERTopic.
Pour chaque topic on regarde :
  - les top-words c-TF-IDF (8 premiers termes)
  - les 6 chunks représentatifs sélectionnés par BERTopic

puis on attribue à chaque topic (i) un nom court humain, (ii) la famille politique dominante d'après `_family_mapping.py`, (iii) une note brève si pertinent.

La sortie finale est sauvegardée dans `outputs/07_bertopic/topic_labels.csv`.

## Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import ast
import re

PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == 'ntbks' else Path.cwd()
TOPIC_INFO    = PROJECT_ROOT / 'outputs' / '07_bertopic' / 'topic_info.csv'
TOPIC_LABELS  = PROJECT_ROOT / 'outputs' / '07_bertopic' / 'topic_labels.csv'
CHUNKS_TOPICS = PROJECT_ROOT / 'outputs' / '07_bertopic' / 'chunks_with_topics.csv'

info = pd.read_csv(TOPIC_INFO)
info['Words'] = info['Representation'].apply(ast.literal_eval)
info = info[info['Topic'] != -1].sort_values('Topic').reset_index(drop=True)

chunks = pd.read_csv(CHUNKS_TOPICS, usecols=['doc_id', 'topic', 'chunk_text'])
print(f"{len(info)} topics retenus (hors HDBSCAN noise) | {len(chunks):,} chunks chargés")
info[['Topic', 'Count']].head(10)

29 topics retenus (hors HDBSCAN noise) | 118,383 chunks chargés


,Topic,Count
0,0,17335
1,1,13330
2,2,8836
3,3,4654
4,4,3164
5,5,1098
6,6,1069
7,7,742
8,8,618
9,9,392


## Outil d'inspection

Fonction utilitaire pour afficher un topic donné : top words + 6 chunks représentatifs (tronqués à 350 caractères pour la lisibilité).

In [2]:
def show_topic(tid: int, n_chunks: int = 5, n_chunk_chars: int = 350, random_state: int = 42):
    row = info[info['Topic'] == tid].iloc[0]
    words = row['Words'][:8]
    sub = chunks.loc[chunks['topic'] == tid].drop_duplicates(subset='doc_id')
    n = min(n_chunks, len(sub))
    docs = sub.sample(n, random_state=random_state)['chunk_text'].tolist()
    print(f"=== Topic {tid}  |  count = {row['Count']}  |  unique docs = {len(sub)} ===")
    print(f"Auto-name: {row['Name']}\n")
    print('Top words:')
    for w in words:
        print(f"  - {w}")
    print(f'\n{n} chunks (1 par doc distinct, sample seed={random_state}) :')
    for i, d in enumerate(docs, 1):
        d = re.sub(r'\s+', ' ', str(d)).strip()[:n_chunk_chars]
        print(f"  [{i}] {d}...\n")

## Inspection topic par topic

Pour chaque topic : on appelle `show_topic(N)`, on lit, et on assigne le label dans `decisions[N]`.

Les décisions sont stockées sous la forme `(label, family_hint, notes)` puis compilées et sauvegardées en fin de notebook.

Les abréviations utilisées : **LO** = Lutte Ouvrière, **LCR** = Ligue Communiste Révolutionnaire, **POE** = Parti Ouvrier Européen, **PLN** = Parti de la Loi Naturelle, **PFN** = Parti des Forces Nouvelles, **PCF** = Parti Communiste Français, **FN** = Front National.

In [3]:
decisions = {}

### Topic 0

In [4]:
show_topic(0)

=== Topic 0  |  count = 17335  |  unique docs = 8855 ===
Auto-name: 0_développement_retraite_die_droit

Top words:
  - développement
  - retraite
  - die
  - droit
  - formation
  - agriculture
  - moyens
  - femmes

5 chunks (1 par doc distinct, sample seed=42) :
  [1] union ouvrière et paysanne pourla démocratie prolétarienne essonne - circonscription godefroy pierre employé suppléant deschamps jacques chercheur travailleuses travailleurs dans notre circonscription nous étions électeurs il ans nous sommes maintenant plus de cette croissance en population pose un tas de problèmes que le capitalisme est incapable ...

  [2] - par l'aménagement concerté au niveau régional du développement industriel artisanal et commercial permettant de créer des emplois nouveaux - par une formation professionnelle adaptée - il faut relancer l'économie - par la relance de la consommation populaire augmentation progressive du smic des prestations familiales lutte contre la hausse des pr...

  [3] le chôm

In [5]:
# salaires, retraite, droits sociaux → programme commun

decisions[0] = ('gauche unie / programme commun', 'socialist_left', 'vocabulaire PS+PCF mainstream')

### Topic 1

In [6]:
show_topic(1)

=== Topic 1  |  count = 13330  |  unique docs = 8505 ===
Auto-name: 1_national_française_majorité_front national

Top words:
  - national
  - française
  - majorité
  - front national
  - front
  - circonscription
  - votez
  - république

5 chunks (1 par doc distinct, sample seed=42) :
  [1] redresse- ment de notre pays vos idées votez christian pierard remplaçant éventuel gérard leleu lepen populaire et nationale liste entente vu les candidats pour nous écrire rue du général clergerie parismaintenant la force d'avenir c'est le front national appel aux français le avril électrices et électeurs m'ont fait confiance - pour promouvoir le ...

  [2] avec -vous république française - ème circonscription du rhône - ler tour martine david le cœur l'action madame mademoiselle monsieur s'il fallait en croire les propos arrogants et revanchards de la droite les discours ironiques voire opportunistes des écolo- gistes les déclarations haineuses de l'extrême-droite le parti socialiste aurait acco

In [7]:
# force d'avenir, redressement national, signature claire FN

decisions[1] = ('FN Le Pen', 'national_right', 'template doc-dup massif')

### Topic 2

In [8]:
show_topic(2)

=== Topic 2  |  count = 8836  |  unique docs = 5155 ===
Auto-name: 2_communiste_communistes_union_voix

Top words:
  - communiste
  - communistes
  - union
  - voix
  - tour
  - mitterrand
  - socialiste
  - candidats

5 chunks (1 par doc distinct, sample seed=42) :
  [1] la droite et au grand patronat que vous n'êtes pas résignés vous laisser faire vous indiquerez françois mitterrand que c'est cette voie là et non celle d'une politique de droite que vous souhaitez voir prendre au pays le juin chaque voix communiste puteaux et neuilly comme au plan national va ainsi être une voix utile pour dire non l'austérité au c...

  [2] mesure où ces progrès contribuent améliorer leur fortune mais dès lors que le progrès doit être social - c'est- -dire profiter la grande masse de la population et en premier lieu ceux qui travaillent - alors là la marche en avant le progrès la révolution leur font peur et tous les moyens sont bons pour les combattre aujourd'hui ceux qui sont au pou...

  [3] avec 

In [9]:
# communistes critiquent 'chèque en blanc' au PS

decisions[2] = ('PCF anti-Mitterrand', 'communist_left', 'post-rupture UG (1977)')

### Topic 3

In [10]:
show_topic(3)

=== Topic 3  |  count = 4654  |  unique docs = 3847 ===
Auto-name: 3_avertissement_député_circonscription_lutte ouvriere

Top words:
  - avertissement
  - député
  - circonscription
  - lutte ouvriere
  - ouvriere
  - candidat
  - bulletin vote
  - demandent

5 chunks (1 par doc distinct, sample seed=42) :
  [1] des hauts fonctionnaires représentons les sans logis au sein de l'administration tous les échelons ministères régions départements communes la collaboration des fonctionnaires avec les hommes de terrain permettra l'amélioration de certains points critiques sur un plan général et l'étude de situation cas par cas d'une part afin d'éviter les abus d'a...

  [2] lacombe est possible le choix est clair pour que vous puissiez voter pour jean lacombe au second tour il faut que vous le placiez en tête dès le premier tour le mars votez utile votez en masse faites voter vos parents vos amis pour jean lacombe candidat du renouveau socialiste pour le renouveau de la circonscription et son 

In [11]:
# travailleuses travailleurs, hostilité droite et méfiance gauche

decisions[3] = ('LO appel travailleurs', 'radical_left', "signature 'avertissement'")

### Topic 4

In [12]:
show_topic(4)

=== Topic 4  |  count = 3164  |  unique docs = 1726 ===
Auto-name: 4_écologie_ecologistes_ecologie_écologistes

Top words:
  - écologie
  - ecologistes
  - ecologie
  - écologistes
  - entente
  - verts
  - entente ecologistes
  - environnement

5 chunks (1 par doc distinct, sample seed=42) :
  [1] entente des ecologistes elections législatives scrutins du et mars - département des bdr - ème circonscription marie-claire mouyrin ans - directrice d'école maternelle - adjointe au maire de mimet dans la ème circonscription l'entente des ecologistes est devenue une réalité de terrain generation ecolo- gie et les verts ont mené une campagne commune...

  [2] république française - élections législatives des et mars imp panorama les nouveaux ecologistes du rassemblement nature et animaux les nouveaux ecologistes du rassemblement nature et animaux les noms de notre candidat et du suppléant se trouvent sur les bulletins de vote comportant notre emblème le trèfle quatre feuilles construisons un 

In [13]:
# entente écologistes verts génération écologie

decisions[4] = ('écolos Verts/GE 1993', 'ecologist', 'coalition écolo 93')

### Topic 5

In [14]:
show_topic(5)

=== Topic 5  |  count = 1098  |  unique docs = 953 ===
Auto-name: 5_président_conseiller_général_maire

Top words:
  - président
  - conseiller
  - général
  - maire
  - suppléant
  - conseil
  - ancien
  - membre

5 chunks (1 par doc distinct, sample seed=42) :
  [1] docteur antoine lacroix et son suppléant éventuel jean-claude denne réformateur membre du conseil politique national du centre démocrate le docteur antoine lacroix est né le mai saint-hilaire allier cinquième fils d'un ouvrier mineur il fut élevé dans une famille très unie marié ans il puise dans son foyer la force morale nécessaire son activité pr...

  [2] de l'essonne depuis il été réélu au er tour en mars la tête de la liste d'union de la gauche avec plus de des voix c'est dire la confiance que ses concitoyens lui accordent andré bussery ans marié une génécologue il est père de enfants spécialiste des problèmes économiques du tiers-monde il un long passé d'action militante la jeunesse etudiante chr...

  [3] centre dé

In [15]:
# conseiller général, maire, légion d'honneur

decisions[5] = ('présentation candidats (CV)', 'boilerplate', 'transversal, biais gaulliste-progrès')

### Topic 6

In [16]:
show_topic(6)

=== Topic 6  |  count = 1069  |  unique docs = 1062 ===
Auto-name: 6_lutte ouvrière_ouvrière_femmes_travailleuses

Top words:
  - lutte ouvrière
  - ouvrière
  - femmes
  - travailleuses
  - travailleuses travailleurs
  - présente
  - candidats
  - arlette laguiller

5 chunks (1 par doc distinct, sample seed=42) :
  [1] élections législatives du mars pourquoi des candidatures lutte ouvrière les candidats de lutte ouvrière sont des candidats révolutionnaires les révolutionnaires vous en avez entendu parler mais vous les connaissez mal la radio la presse la télévision les grands partis tous s'unissent pour défigurer leur image déformer leurs propos par exemple on di...

  [2] élections législatives de mars circonscription du nord marie-cécile pilatus institutrice ans présentée par lutte ouvrière suppléante brigitte gaillet professeur ans travailleuses travailleurs lutte ouvrière le parti que je représente ici présente des candidats dans toutes les circonscrip- tions du pays et parmi ces 

In [17]:
# présenté par lutte ouvrière + suppléant

decisions[6] = ('LO investiture', 'radical_left', 'template formel')

### Topic 7

In [18]:
show_topic(7)

=== Topic 7  |  count = 742  |  unique docs = 455 ===
Auto-name: 7_devons_patrons_prenne_bénéfices

Top words:
  - devons
  - patrons
  - prenne
  - bénéfices
  - bourgeoisie
  - patronat
  - prenne richesse
  - bourgeoisie fortunes

5 chunks (1 par doc distinct, sample seed=42) :
  [1] que de contri- buer la réalisation de cette unité il n' aucune fatalité au chomage la précarité on ose nous dire qu'il n' pas d'argent pour l'emploi les salaires l'école la sécurité sociale et les retraites et pourtant ils sont tous d'accord pour - dépenser millions de francs pour le raid d'un seul avion sur l'irak - verser milliards dans la poche ...

  [2] travailleurs immigrés bout de souffle et d'imagination le pouvoir promet aux français la prospérité pour condition bien entendu que les travailleurs acceptent d'ici là la semaine de travail la plus longue d'europe les salaires les plus bas et l'emballement des prix ça n'est pas possible ça n'est pas pensable c'est pourquoi il existe une telle volon.

In [19]:
# fortunes bourgeoisie, faire payer les patrons

decisions[7] = ('LO patronat', 'radical_left', '')

### Topic 8

In [20]:
show_topic(8)

=== Topic 8  |  count = 618  |  unique docs = 405 ===
Auto-name: 8_hommes politiques_politiques_châteaux_vivent

Top words:
  - hommes politiques
  - politiques
  - châteaux
  - vivent
  - payent journalistes
  - argent vivent
  - journalistes hommes
  - vivent châteaux

5 chunks (1 par doc distinct, sample seed=42) :
  [1] pas l'etat dépense des sommes énormes l'entretien d'une armée d'une police d'une administration dont le seul but est de tromper les travailleurs de les maintenir dans l'ignorance de son gaspillage et de sa gabegie et de réprimer toute tentative de faire connaître la vérité et de s'opposer sa politique les travailleurs en ont assez de ce gouvernemen...

  [2] hommes politiques qui les aident on découvre alors qu'ils en ont des propriétés des châteaux des comptes en banque qu'il en des pots de vin qui s'échangent des millions de francs rien qu'en pourboires ou des chefs de parti de la droite qui peuvent se payer ou se faire offrir un réveillon de francs millions de ce

In [21]:
# châteaux, pots-de-vin, journalistes

decisions[8] = ('LO corruption', 'radical_left', '')

### Topic 9

In [22]:
show_topic(9)

=== Topic 9  |  count = 392  |  unique docs = 392 ===
Auto-name: 9_devons_comptes_patronat_arrêter

Top words:
  - devons
  - comptes
  - patronat
  - arrêter
  - route faudra
  - train préparer
  - gages garanties
  - violence sordide

5 chunks (1 par doc distinct, sample seed=42) :
  [1] nous refusons de gérer la crise c'est pourquoi nous vous demandons de vous battre avec nous autour de cinq grands engagements pour en sortir remettre l'argent au service de la production agricole et industrielle arrêter l'usure financière et la spéculation boursière ou immobilière qui détruisent l'emploi non aux quotas non au démantèlement de l'ind...

  [2] ne devons pas mendier nous devons viser les entreprises la tête le patronat au portefeuille les riches la bourse oui il peut avoir un nouveau juin ou un nouveau mai mais cette fois nous ne nous laisserons pas arrêter en route il nous faudra des gages et des garanties il n' que la peur qui fera reculer le patronat et nous n'avons pas nous poser de

In [23]:
# réf juin 36 et mai 68

decisions[9] = ('LO mai 68', 'radical_left', '')

### Topic 10

In [24]:
show_topic(10)

=== Topic 10  |  count = 332  |  unique docs = 288 ===
Auto-name: 10_rpr_udf_rpr udf_ps

Top words:
  - rpr
  - udf
  - rpr udf
  - ps
  - rétablir
  - pc
  - immigration
  - carte séjour

5 chunks (1 par doc distinct, sample seed=42) :
  [1] ps en ile-de-france au rpr rpr-udf ils vous trompent ils vous disent qu'ils sont contre l'immigration en réalité avec le ps et le pc le rpr et l'udf ont voté la carte de séjour de ans renouvelable pour les immigrés qu'ils vont réformer le code de la nationalité ils l'avaient déjà promis en ils n'en ont rien fait qu'ils vont rétablir la sécurité les...

  [2] ils n'en ont rien fait qu'ils vont rétablir la sécurité les élus du rpr et de l'udf se refu- sent encore aujourd'hui rétablir la peine de mort qu'ils vont sauver l'agriculture le rpr et l'udf ont approuvé la pac et ont dit oui au traité de maastricht rpr-ps-udf-pc tous responsables tous coupables immigration chômage impôts insécurite injustices cor...

  [3] délits et de crimes - millions d'im

In [25]:
# ils vous trompent, carte de séjour, peine de mort

decisions[10] = ('FN tous coupables RPR-UDF-PS', 'national_right', 'attaque coalition mainstream')

### Topic 11

In [26]:
show_topic(11)

=== Topic 11  |  count = 248  |  unique docs = 248 ===
Auto-name: 11_animaux_nature animaux_corridas_ruffi marseille

Top words:
  - animaux
  - nature animaux
  - corridas
  - ruffi marseille
  - ruffi
  - rue ruffi
  - programmes faveur
  - complète code

5 chunks (1 par doc distinct, sample seed=42) :
  [1] disposant du droit de véto ainsi tout risque de pollution sera écarté pour les animaux interdiction de la vivisection pratiquée uniquement pour des questions d'argent amélioration de leurs conditions d'existence lutte contre les abus de la chasse les corridas les massacres dans les abattoirs les abandons et les mauvais traitements bulletin - répons...

  [2] du droit de véto ainsi tout risque de pollution sera écarté pour les animaux interdiction de la vivisection pratiquée uniquement pour des questions d'argent amélioration de leurs conditions d'existence lutte contre les abus de la chasse les corridas les massacres dans les abattoirs les abandons et les mauvais traitements bull

In [27]:
# corridas, vivisection, rue ruffi marseille

decisions[11] = ('défense animaux', 'ecologist', 'petit parti animaliste')

### Topic 12

In [28]:
show_topic(12)

=== Topic 12  |  count = 247  |  unique docs = 178 ===
Auto-name: 12_afrique_illusion_sortir_autour grands

Top words:
  - afrique
  - illusion
  - sortir
  - autour grands
  - soviétique
  - pouvons accepter
  - alliés
  - afrique esprit

5 chunks (1 par doc distinct, sample seed=42) :
  [1] immigré lorsque ceux-ci ne résident pas sur le territoire français outre cette charte nous proposons une mesure ponctuelle immédiate que les fonds destinés verser chaque immigré retournant dans son pays soient utilisés pour compléter l'indemnisation des français rapatriés d'afrique du nord déjà immigrés ont perçu cette indemnité scandaleuse et inju...

  [2] travailleur immigré lorsque ceux-ci ne résident pas sur le territoire français outre cette charte nous proposons une mesure ponctuelle immédiate que les fonds destinés verser chaque immigré retournant dans son pays soient utilisés pour compléter l'indemnisation des français rapatriés d'afrique du nord déjà immigrés ont perçu cette indemnité sc

In [29]:
# programme hadès, illusion russie, réf hitler/années 30

decisions[12] = ('POE anti-soviétique', 'radical_left', 'réseau LaRouche')

### Topic 13

In [30]:
show_topic(13)

=== Topic 13  |  count = 234  |  unique docs = 234 ===
Auto-name: 13_comptes_banque_comptes banque_employés

Top words:
  - comptes
  - banque
  - comptes banque
  - employés
  - avoirs
  - propriétés
  - collectivement courant
  - publier tendance

5 chunks (1 par doc distinct, sample seed=42) :
  [1] peu plus qu'on ouvre les comptes en banque patronaux quand les patrons pleurent misère ils veulent qu'on les croie sur parole mais qu'ils ouvrent donc leurs livres de comptes et pas seulement ceux des entreprises qu'ils dévoilent leurs comptes en banque personnels ceux de leur famille voire de leurs maîtresses qu'ils nous disent quels sont leurs av...

  [2] peu plus qu'on ouvre les comptes en banque patronaux quand les patrons pleurent misère ils veulent qu'on les croie sur parole mais qu'ils ouvrent donc leurs livres de comptes et pas seulement ceux des entreprises qu'ils dévoilent leurs comptes en banque personnels ceux de leur famille voire de leurs maîtresses qu'ils nous disent quel

In [31]:
# ouvrir les livres de comptes patronaux

decisions[13] = ('LO comptes patrons', 'radical_left', '')

### Topic 14

In [32]:
show_topic(14)

=== Topic 14  |  count = 234  |  unique docs = 230 ===
Auto-name: 14_changent_voulons_économie post_criaient

Top words:
  - changent
  - voulons
  - économie post
  - criaient
  - trafic influences
  - gouvernement bourgeoisie
  - utiliser peur
  - misère vieux

5 chunks (1 par doc distinct, sample seed=42) :
  [1] arrière et pour infléchir la politique du nou- veau septennat vers l'économie post-industrielle les recettes du passé ne guériront pas une économie malade du gaspillage et de la croissance non la fuite en avant la france doit refuser la guerre écono- mique où le plus fort écrase le plus faible où les pays riches ruinent le tiers monde nous proposon...

  [2] est en paix partout dans le monde nos finances sont saines et notre economie est en nette progression de l'avis des experts étrangers la france sera dans quelques années la troisième puissance économique du monde après l'amérique et le japon loin devant les pays de l'est cette situation favorable permis au gouvernement 

In [33]:
# ligue communiste section française quatrième internationale

decisions[14] = ('LCR', 'radical_left', 'trotskyste 4e Internationale')

### Topic 15

In [34]:
show_topic(15)

=== Topic 15  |  count = 230  |  unique docs = 157 ===
Auto-name: 15_maire_adjoint_municipal_conseiller municipal

Top words:
  - maire
  - adjoint
  - municipal
  - conseiller municipal
  - conseiller
  - eta
  - st
  - di

5 chunks (1 par doc distinct, sample seed=42) :
  [1] mouligneau - maire de mallerey gilbert py - maire de chemenot andré vialait - adjoint cray et charnay gilbert campy - conseiller municipal lavigny daniel tisserand - conseiller municipal fontainebrux andré camelin - conseiller municipal briod vu les candidats - iov...

  [2] carentan fossard marcel inspecteur des et t militant syndical saint-lo gauthier albert mécanicien maire-adjoint saint-andré-de- bohon guignier henriette journaliste saint-lo guillaume mlle conseillère municipale airel heitzmann jean-claude artisan saint-lo houlgate jean-marie artisan carentan hurel roger et robert commerçants carentan lalouelle ala...

  [3] dugun eta maite duzuen herriari buruz kanpaina ezberdina egin nahi izan dugu hunkitz

In [35]:
# énumération de maires/adjoints par commune

decisions[15] = ('listes élus signataires', 'boilerplate', '')

### Topic 16

In [36]:
show_topic(16)

=== Topic 16  |  count = 170  |  unique docs = 170 ===
Auto-name: 16_etat_services publics_publics etat_services

Top words:
  - etat
  - services publics
  - publics etat
  - services
  - publics
  - financer
  - payer
  - impôts

5 chunks (1 par doc distinct, sample seed=42) :
  [1] c'est dans la poche des contri- buables que l'etat le prend ce sont nos impôts pris directement sur notre salaire ou indirectement sur tout ce que nous achetons en tant que consommateurs qui aident les plus puissants devenir encore plus riches le rôle de l'etat ce serait de financer les services publics qui servent tout le monde l'education natio- ...

  [2] c'est dans la poche des contri- buables que l'etat le prend ce sont nos impôts pris directement sur notre salaire ou indirectement sur tout ce que nous achetons en tant que consommateurs qui aident les plus puissants devenir encore plus riches le rôle de l'etat ce serait de financer les services publics qui servent tout le monde l'education natio- ...

In [37]:
# État au service du capital, contribuables

decisions[16] = ('LO impôts', 'radical_left', '')

### Topic 17

In [38]:
show_topic(17)

=== Topic 17  |  count = 165  |  unique docs = 165 ===
Auto-name: 17_bulletin vote_bulletin_peuple algérien_guerre radicaux

Top words:
  - bulletin vote
  - bulletin
  - peuple algérien
  - guerre radicaux
  - gauche bourgeois
  - bourgeois banquiers
  - banquiers quête
  - négociation peuple

5 chunks (1 par doc distinct, sample seed=42) :
  [1] maintenant il affirme que les travailleurs travailleraient avec plus d'ardeur sous la direction d'un gouvernement d'union populaire - le parti socialiste qui s'est compromis dans de nombreux gouvernements de la quatrième et même de la cinquième république puisque c'est guy mollet qui est allé chercher de gaulle en et fut un temps un de ses ministre...

  [2] maintenant il affirme que les travailleurs travailleraient avec plus d'ardeur sous la direction d'un gouvernement d'union populaire - le parti socialiste qui s'est compromis dans de nombreux gouvernements de la quatrième et même de la cinquième république puisque c'est guy mollet qui est 

In [39]:
# PS compromis dans IVe République

decisions[17] = ('LO critique PS', 'radical_left', '')

### Topic 18

In [40]:
show_topic(18)

=== Topic 18  |  count = 151  |  unique docs = 92 ===
Auto-name: 18_maharishi_criminalité_compléter système_compléter

Top words:
  - maharishi
  - criminalité
  - compléter système
  - compléter
  - système
  - science védique
  - védique maharishi
  - produira baisse

5 chunks (1 par doc distinct, sample seed=42) :
  [1] atteindre ce résultat le programme de mt- sidhi cette simple mesure produira une baisse instantanée des problèmes notamment de la criminalité et du chômage et stimulera la reprise economie minimum pour l'etat milliards de francs dans les premières années sur cette base de cohérence nécessaire nous réussirons le redéveloppement de la france par un p...

  [2] et que seuls des intérêts privés ou partisans ont empêché leur mise en place jusqu' ce jour t tenant compte de notre expérience nous pouvons affirmer que notre programme entraînera la baisse de par an du chômage de des maladies de de la récidive de par an de la criminalité l'accroîssement de sur ans de la product

In [41]:
# mt-sidhi, science védique, criminalité

decisions[18] = ('PLN Maharishi', 'niche', 'Parti de la Loi Naturelle')

### Topic 19

In [42]:
show_topic(19)

=== Topic 19  |  count = 149  |  unique docs = 149 ===
Auto-name: 19_petits_petits commerçants_hommes politiques_milliers

Top words:
  - petits
  - petits commerçants
  - hommes politiques
  - milliers
  - socialiste petits
  - commerçants
  - politiques
  - commerçants saignés

5 chunks (1 par doc distinct, sample seed=42) :
  [1] les petits commerçants les artisans doivent eux aussi voter contre les hommes politiques de la droite qui ont surimposé les petits pour mieux subventionner les gros sous leur gouvernement plus du quart de la paysannerie été chassé de ses terres des centaines de milliers de jeunes ont dû quitter les campagnes faute de pouvoir vivre des dizaines de m...

  [2] petits commerçants les artisans doivent eux aussi voter contre les hommes politiques de la droite qui ont surimposé les petits pour mieux subventionner les gros sous leur gouvernement plus du quart de la paysannerie été chassé de ses terres des centaines de milliers de jeunes ont dû quitter les campagne

In [43]:
# artisans, paysannerie, classes moyennes inf

decisions[19] = ('LO petits commerçants', 'radical_left', '')

### Topic 20

In [44]:
show_topic(20)

=== Topic 20  |  count = 140  |  unique docs = 67 ===
Auto-name: 20_coup_temps vivre_oiseaux_change relever

Top words:
  - coup
  - temps vivre
  - oiseaux
  - change relever
  - supprimer hierarchies
  - hierarchies
  - inégalités supprimer
  - ca

5 chunks (1 par doc distinct, sample seed=42) :
  [1] le fro- mage ont encore augmente il fau- dra bientôt une brouette pour faire bon marche et la s n c et l'essence iln' que les salaires qui ne bougent pas c'est une honte ils m'ont licencié malgré mes quinze ans d'ancienneté j'étais pourtant bien noté jamais d'absence on croit que ça n'arrive qu'aux autres et quand ca nous tombe sur la tete cafait m...

  [2] la s n c et l'essence iln'ya que les salaires qui ne bougent pas c'est une honte ils m'ont licencié malgré mes quinze ans d'ancienneté j'étais pourtant bien noté jamais d'absence on croit que ça n'arrive qu'aux autres et quand ca nous tombe sur la tête ca fait mal c'est les poubelles ourien si tu n'es pas content re- tourne dans ton

In [45]:
# dialogue grand-père/petit-fils, oiseaux, bureau du chômage

decisions[20] = ('tract poétique (chômage/écologie)', 'mixed', 'format atypique')

### Topic 21

In [46]:
show_topic(21)

=== Topic 21  |  count = 113  |  unique docs = 113 ===
Auto-name: 21_technologies_petits_recherche_guerre

Top words:
  - technologies
  - petits
  - recherche
  - guerre
  - circonstances
  - résistance totalitarisme
  - garantie résistance
  - nécessaires guerre

5 chunks (1 par doc distinct, sample seed=42) :
  [1] faut ouvrir nos enfants des emplois qui les enthousiasment aux frontières de la recherche non des petits boulots ou le chômage organiser l'application de ces technologies nouvelles au domaine militaire elles sont la fois garantie d'une résistance au totalitarisme et facteur de gains de productivité par leurs retombées dans l'économie civile au-delà...

  [2] jugez digne je pourrai vous représenter valablement et utilement au parlement dire tout ce que vous aurez dire aux politiciens et faire entendre votre voix quelles que soient les circonstances politiques il faut voter contre la droite il faut que les travailleurs le mars votent massivement contre la droite qui fait re

In [47]:
# résistance totalitarisme, technologies militaires

decisions[21] = ('POE technologies / défense', 'radical_left', '')

### Topic 22

In [48]:
show_topic(22)

=== Topic 22  |  count = 90  |  unique docs = 90 ===
Auto-name: 22_pouvons accepter_accepter_pouvons_illusion

Top words:
  - pouvons accepter
  - accepter
  - pouvons
  - illusion
  - capacité produire
  - démagogie extrémiste
  - parcs schtroumpfs
  - laminée

5 chunks (1 par doc distinct, sample seed=42) :
  [1] temps parallèlement la domination du profit financier on propage une idéologie du gain rapide le jeu est partout tacotac télémago pmu sous l'illusion de la société post-industrielle et des parcs schtroumpfs c'est notre capacité de produire et de nous défendre qu'on détruit car dans un second temps les dirigeants russes espèrent que cette europe de ...

  [2] temps parallèlement la domination du profit financier on propage une idéologie du gain rapide le jeu est partout tacotac télémago pmu sous l'illusion de la société post-industrielle et des parcs schtroumpfs c'est notre capacité de produire et de nous défendre qu'on détruit car dans un second temps les dirigeants russes e

In [49]:
# parcs schtroumpfs, profit financier

decisions[22] = ('POE post-industriel', 'radical_left', 'critique financiarisation')

### Topic 23

In [50]:
show_topic(23)

=== Topic 23  |  count = 90  |  unique docs = 90 ===
Auto-name: 23_cartels_europe_version_production

Top words:
  - cartels
  - europe
  - version
  - production
  - accepte
  - cartels assurance
  - blocage production
  - alimentaire veut

5 chunks (1 par doc distinct, sample seed=42) :
  [1] parti ouvrier européen siège rue nollet paris elections législatives du juin pour l'europe de la production contre l'europe des cartels si rien n'est fait pour l'en sortir la france est prise au piège nous allons tout droit une europe de dominée par les cartels de l'assurance de la banque et de l'agro-alimentaire cela veut dire fermeture d'usines p...

  [2] parti ouvrier européen siège rue nollet paris elections législatives du juin pour l'europe de la production contre l'europe des cartels si rien n'est fait pour l'en sortir la france est prise au piège nous allons tout droit une europe de dominée par les cartels de l'assurance de la banque et de l'agro-alimentaire cela veut dire fermeture d'u

In [51]:
# europe des cartels vs production

decisions[23] = ('POE Europe production', 'radical_left', '')

### Topic 24

In [52]:
show_topic(24)

=== Topic 24  |  count = 84  |  unique docs = 84 ===
Auto-name: 24_formation_enfant_cas_réhabiliter défendre

Top words:
  - formation
  - enfant
  - cas
  - réhabiliter défendre
  - clubs réhabiliter
  - familles françai
  - intéressés mettre
  - actifs mères

5 chunks (1 par doc distinct, sample seed=42) :
  [1] l'homme et du citoyen notamment mise en place du tiers temps éducatif formation générale l'etat for- mation professionnelle au monde écono- mique formation artistique dans les aca- démies régionales et les ateliers munici- paux et formation sportive dans les clubs - réhabiliter et défendre la famille revenu parental pour les familles françai- ses d...

  [2] trompes ou tout autre inter- vention médicale et chirurgicale dans les cas sui- vants - après le troisième enfant dans le cas général - sans limitation après ans - après avis d'une commission médico-sociale dans les cas de déficience physique ou mentale définitive la stérilisation masculine volontaire - après acceptation 

In [53]:
# tiers temps éducatif, formation professionnelle

decisions[24] = ('éducation / formation', 'mixed', 'ambigu (peut-être FN/PFN)')

### Topic 25

In [54]:
show_topic(25)

=== Topic 25  |  count = 78  |  unique docs = 78 ===
Auto-name: 25_violence_révolution_bourgeoisie_luttes

Top words:
  - violence
  - révolution
  - bourgeoisie
  - luttes
  - libération chili
  - travailleurs arrachent
  - socialisme obtient
  - arrachent revendications

5 chunks (1 par doc distinct, sample seed=42) :
  [1] nous dit-on vous n'avez que les mots de lutte de violence et de révolution la bouche avec vous c'est toujours le tout ou rien c'est seulement la leçon de l'expérience c'est par les luttes que les travailleurs arrachent leurs revendications la bourgeoisie c'est aussi dans leurs luttes qu'ils commencent prendre les choses en main entrevoir qu'une aut...

  [2] nous dit-on vous n'avez que les mots de lutte de violence et de révolution la bouche avec vous c'est toujours le tout ou rien c'est seulement la leçon de l'expérience c'est par les luttes que les travailleurs arrachent leurs revendications la bourgeoisie c'est aussi dans leurs luttes qu'ils commencent prendre 

In [55]:
# luttes révolutionnaires, libération chili

decisions[25] = ('LCR Chili', 'radical_left', 'internationalisme 70s')

### Topic 26

In [56]:
show_topic(26)

=== Topic 26  |  count = 78  |  unique docs = 78 ===
Auto-name: 26_fondée_loi_accord loi_potentiel

Top words:
  - fondée
  - loi
  - accord loi
  - potentiel
  - apprentissage
  - créant cohérence
  - investissement créant
  - cohérence redonnant

5 chunks (1 par doc distinct, sample seed=42) :
  [1] système d'éducation actuel par l'introduction de la connaissance du potentiel créateur infini de la nature qui promeut une pensée une parole et une action en accord avec la loi naturelle et l'individu augmente l'efficacité la réusssite la satisfaction et le bonheur de éliminer le chômage par la formation des chômeurs utiliser leur plein potentiel e...

  [2] potentiel créateur infini de la nature qui promeut une pensée une parole et une action en accord avec la loi naturelle et l'individu augmente l'efficacité la réusssite la satisfaction et le bonheur de emploi economie solidarité éliminer le chômage par la formation des chômeurs utiliser leur plein potentiel et par la création d'emplois

In [57]:
# potentiel créateur infini de la nature

decisions[26] = ('PLN loi naturelle', 'niche', 'sous-thème PLN')

### Topic 27

In [58]:
show_topic(27)

=== Topic 27  |  count = 77  |  unique docs = 77 ===
Auto-name: 27_charte immigration_concerne_charte_immigration

Top words:
  - charte immigration
  - concerne
  - charte
  - immigration
  - suisse
  - immigration soumise
  - soumise
  - mesure

5 chunks (1 par doc distinct, sample seed=42) :
  [1] soumise referendum - un référendum sur l'immigration comme cela déjà été fait en suisse nous préconisons cette mesure parce que ce problème prend chaque jour une acuité accrue et qu'il concerne tous les français si une soluti n n'est pas trouvée d'urgence comme en suisse ou en notre pays risque de connaître des troubles graves semblables ceux qui s...

  [2] des forces nouvelles bd de sébastopol paris tél vu le candidatquelques points de programme du parti des forces nouvelles pour une charte de l'immigration soumise referendum - un référendum sur l'immigration comme cela déjà été fait en suisse nous préconisons cette mesure parce que ce problème prend chaque jour une acuité accrue et qu'i

In [59]:
# charte immigration, référendum suisse

decisions[27] = ('FN référendum immigration', 'national_right', '')

### Topic 28

In [60]:
show_topic(28)

=== Topic 28  |  count = 63  |  unique docs = 63 ===
Auto-name: 28_éducation_mesures obligatoires_invalidité vieillesse_maladie invalidité

Top words:
  - éducation
  - mesures obligatoires
  - invalidité vieillesse
  - maladie invalidité
  - facultatives relevant
  - complémentaires facultatives
  - sociale maladie
  - obligatoires protection

5 chunks (1 par doc distinct, sample seed=42) :
  [1] promulgué une loi sur les plus-values qui constitue un véritable impôt sur le capital alors que celui-ci est érodé quotidiennement par l'inflation on admet que des préposés du secteur public organisent des grêves qui permettent une minorité communiste de paralyser les entreprises rien n' été fait pour mettre fin au scandale de la loi edgar faure qu...

  [2] parlement soit qu'ils n'imposent pas de charges publiques excessives soit qu'ils permettent même d'appréciables éco- nomies budgétaires savoir dans divers domaines abrogation de la loi d'orientation dite loi edgar faure qui désagrégé nos 

In [61]:
# abrogation loi edgar faure, anti-marxisme

decisions[28] = ('PFN anti-loi Faure', 'national_right', 'extrême-droite éducation')

## Compilation et sauvegarde

On consolide les décisions dans un DataFrame aligné sur l'ordre de `topic_info.csv`, puis on écrit `topic_labels.csv` avec les colonnes `topic_id, auto_name, label, family_hint, notes`.

In [62]:
out = pd.DataFrame({
    'topic_id':    info['Topic'].astype(int),
    'auto_name':   info['Name'],
    'label':       info['Topic'].map(lambda t: decisions[t][0]),
    'family_hint': info['Topic'].map(lambda t: decisions[t][1]),
    'notes':       info['Topic'].map(lambda t: decisions[t][2]),
})

out.to_csv(TOPIC_LABELS, index=False, encoding='utf-8-sig')
print(f'Wrote {len(out)} labels to {TOPIC_LABELS}')
out[['topic_id', 'label', 'family_hint']]

Wrote 29 labels to /Users/brianramesh/Documents/NLP_PROJET/Topic_modeling_nlp/outputs/07_bertopic/topic_labels.csv


,topic_id,label,family_hint
0,0,gauche unie / programme commun,socialist_left
1,1,FN Le Pen,national_right
2,2,PCF anti-Mitterrand,communist_left
3,3,LO appel travailleurs,radical_left
4,4,écolos Verts/GE 1993,ecologist
5,5,présentation candidats (CV),boilerplate
6,6,LO investiture,radical_left
7,7,LO patronat,radical_left
8,8,LO corruption,radical_left
9,9,LO mai 68,radical_left


## Récapitulatif par famille

Vérification rapide de la distribution des familles attribuées.

In [63]:
out['family_hint'].value_counts()

family_hint
radical_left      15
national_right     4
ecologist          2
boilerplate        2
niche              2
mixed              2
socialist_left     1
communist_left     1
Name: count, dtype: int64